In [2]:
import numpy 

import torch  
import torch.nn as nn    
 


## Requirment 
1.  We will be working on a system in free space, so the V(x) = 0.
 
2. The predicted solution must satisfy a symmetry constraint (which may need to be enforced, e.g., in the decoder);

3. the solution should satisfy a prescribed (L^2) norm, i.e., |\psi|^2 = N, where N depends on parameters (i.e., it is not an independent parameter). 


In [82]:
l =    nn.ConvTranspose1d(
                    16,
                    32, 
                    kernel_size=4,
                    stride=2,
                    padding=1
                )

In [87]:
print(latent_data.shape )

torch.Size([10, 16, 36])


In [88]:
out = l(latent_data)

In [91]:
l =    nn.ConvTranspose1d(
                    32,
                    64, 
                    kernel_size=4,
                    stride=2,
                    padding=1
                ) 

In [92]:
l(out).shape

torch.Size([10, 64, 144])

In [ ]:
class decoder(nn.Module): 
    
    def __init__(self,  latent_dim, hidden_conv_dims, output_sol_dim):
        super().__init__()
        self.latent_dimension = latent_dim 
        
    
        dims = [latent_dim] + hidden_conv_dims

        layers = []

        for i in range(len(dims) - 1):
            layers.append(
                nn.ConvTranspose1d(
                    dims[i],
                    dims[i + 1],
                    kernel_size=4,
                    stride=2,
                    padding=1
                )
            )
            layers.append(nn.ReLU())

       
        layers.append(
            nn.Conv1d(dims[-1], output_sol_dim, kernel_size=1)
        )

        self.model = nn.Sequential(*layers)

        print(self.model)
    
    def enforce_even_symmetry(self, y):
        """
        y: (B, C, L)
        """
        y_flip = torch.flip(y, dims=[-1])   # reverse spatial dimension
        return 0.5 * (y + y_flip)
        
    
    def enforce_l2_norm(self, y, N=1.0, eps=1e-12):
        """
        y: (B, C, L)
        N: desired L2 norm squared
        """
        # compute L2 norm squared over spatial dim
        norm_sq = torch.sum(y ** 2, dim=-1, keepdim=True)  # (B, C, 1)

        # avoid division by zero
        scale = torch.sqrt(N / (norm_sq + eps))

        return y * scale
        

        
    def forward(self, x):
        output = self.model(x)
        # enforce the symmetry 
        
        output_symmetry = self.enforce_even_symmetry(output) 
        
        output_symmetry_normalized = self.enforce_l2_norm( output_symmetry )
         
        return output_symmetry_normalized.permute(0, 2, 1)
            

In [ ]:
class latent_models(nn.Module):
    
    def __init__(self, input_dim, hidden_dims,
                 latent_feature_dim , n_latent, 
                 activation=nn.ReLU, output_activation=None):
        super().__init__()
        self.n_latent = n_latent
        self.latent_feature_dim = latent_feature_dim 
        self.output_dim = n_latent  * latent_feature_dim 
        layers = []
        
        dims = [input_dim] + list(hidden_dims) + [self.output_dim]
      
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i+1]))
            if i < len(dims) - 2:
                layers.append(activation())
        if output_activation is not None:
            layers.append(output_activation())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # x is the (B, number_of_parameters )
        output = self.net(x) 
        
        output = output.view(-1, self.latent_feature_dim, self.n_latent)
        return output 

In [120]:
config = {
    "num_layers": 4,
    "hidden_dims": [128, 128, 128], 
    "latent_feature_dim":16,
    "n_latent":16,
    
    "hidden_conv_dims": [128, 128, 128, 128, ],

    
    "activation":nn.ReLU,
    "output_activation": None, 
    "learning_rate": 1e-3,

    "batch_size": 32,
    "epochs": 100,
    "weight_decay": 1e-5,
    "dropout": 0.1,


    # data
    "input_dim": 2,
    "output_sol_dim": 1
    
    
  
} 

In [121]:
model = latent_models(input_dim = config["input_dim"], 
                      hidden_dims= config["hidden_dims"],
                      latent_feature_dim=  config["latent_feature_dim"], 
                      n_latent= config["n_latent"],  
                      activation= config["activation"], 
                      output_activation=config["output_activation"]  )

In [122]:
import numpy as np 
x = torch.randn(10, 2)
latent_data = model(x)



In [123]:
latent_data.shape

torch.Size([10, 16, 16])

In [131]:
decoder_model = decoder(latent_dim = config["latent_feature_dim"],
         hidden_conv_dims = config["hidden_conv_dims"],
         output_sol_dim = config["output_sol_dim"])

Sequential(
  (0): ConvTranspose1d(16, 128, kernel_size=(4,), stride=(2,), padding=(1,))
  (1): ReLU()
  (2): ConvTranspose1d(128, 128, kernel_size=(4,), stride=(2,), padding=(1,))
  (3): ReLU()
  (4): ConvTranspose1d(128, 128, kernel_size=(4,), stride=(2,), padding=(1,))
  (5): ReLU()
  (6): ConvTranspose1d(128, 128, kernel_size=(4,), stride=(2,), padding=(1,))
  (7): ReLU()
  (8): Conv1d(128, 1, kernel_size=(1,), stride=(1,))
)


In [ ]:
output = decoder_model(latent_data)

torch.Size([10, 1, 256])

In [ ]:
import torch
import torch.nn as nn


# -------------------------
# TRAIN FUNCTION
# -------------------------
def train(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for x, y in dataloader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        pred = model(x)
        loss = criterion(pred, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


# -------------------------
# EVALUATION FUNCTION
# -------------------------
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)

            pred = model(x)
            loss = criterion(pred, y)

            total_loss += loss.item()

    return total_loss / len(dataloader)


# -------------------------
# TRAINER (FIXED)
# -------------------------
def trainer(model, train_loader, val_loader, config, epoch_save, logs, device):

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"]
    )

    # ✅ scheduler FIXED (simple and standard)
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=10,
        gamma=0.95
    )

    criterion = nn.MSELoss()

    best_val = float("inf")

    for epoch in range(config["epochs"]):

        # -----------------
        # train
        # -----------------
        train_loss = train(
            model, train_loader, optimizer, criterion, device
        )
        # -----------------
        # scheduler step
        # -----------------
        scheduler.step()

        if logs is not None:
            logs.log({
                "epoch": epoch,
                "train_loss": train_loss,

            })

        # -----------------
        # save best model
        # -----------------
        if val_loss < best_val:
            best_val = val_loss
            torch.save(
                model.state_dict(),
                f"./checkpoints/best_model.pt"
            )

        # -----------------
        # periodic saving
        # -----------------
        if epoch % epoch_save == 0:
            torch.save(
                model.state_dict(),
                f"./checkpoints/model_epoch_{epoch}.pt"
            )
         
            # -----------------
            # validate EVERY epoch (important fix)
            # -----------------
            val_loss = evaluate(
                model, val_loader, criterion, device
            )
            logs.log({
                "epoch": epoch,
                "val_loss": val_loss,

            })
        
    return model



In [ ]:
def main():
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = YourModel().to(device)

    # fake loaders (replace later)
    train_loader = ...
    val_loader = ...

    config = {
        "learning_rate": 1e-3,
        "weight_decay": 1e-5,
        "epochs": 10
    }

    trainer(model, train_loader, val_loader, config, device)



In [ ]:
if "__name__" == "main": 

config: 


latent_models()